# ses-studio

Kendi sesinizin klonuyla Türkçe seslendirme, Google Colab GPU'sunda (VoxCPM2 + Whisper). Drive gerekmez.

1. **Çalışma zamanı → Çalışma zamanı türünü değiştir →** bir GPU seçin (A100 önerilir; L4 ve T4'te de çalışır, daha yavaş).
2. Aşağıdaki hücreyi çalıştırın (yaklaşık 3 dk). Hücrenin altında sayfa adresi, kullanıcı adı (`ses`) ve şifre çıkar.
3. Sayfada: **Ses profili** (`.wav` ya da `.zip`) + **Metin** (düz metin `.txt` ya da `seslendirme.md` tablosu) → **Üret** → dinleyin, beğenmediğinizi **Sorunlu** işaretleyin → **İşaretlileri yeniden üret** → **Onayla ve birleştir** → `ses.wav` indirin.
4. Bitince **Oturumu kapat**. Oturum kapanınca işler silinir; taşımak için **İşi indir (zip)**, yeni oturumda **İş yükle**.

**Kendi ses profiliniz:** sessiz oda (telefon yeter) · anlatır gibi konuşun · yaklaşık 2 dk kayıt · WAV, tek kanal, 48 kHz, baş/son sessizliği kesilmiş (`ffmpeg -i kayit.m4a -ac 1 -ar 48000 referans.wav`) · Ses profili alanına yükleyin · önce kısa bir metinle deneyin.

⚠️ Yalnızca kendi sesinizi ya da açıkça izin veren birinin sesini klonlayın; ürettiğiniz sesin yapay zekâyla yapıldığını belirtin.

Model: VoxCPM2 (OpenBMB, Apache 2.0). Paket: 2026-11-08 22:16 · 48 proje dosyası


In [ ]:
# @title ses-studio · Başlat  { display-mode: "form" }
# Çalıştır (▶). Birkaç dakika sonra aşağıda sayfa adresi, kullanıcı adı ve şifre çıkar; sonrası sayfada.
# Sayfa açık kaldığı sürece bu hücre çalışır. 30 dk kullanılmazsa oturum kendiliğinden kapanır (compute unit harcanmaz).
SIFRE = ""  # @param {type:"string"}
# boş bırakılırsa her açılışta yeni şifre üretilir
MODEL_SAYISI = 0  # @param {type:"integer"}
# 0 = GPU belleğine göre otomatik (20 GB başına bir model: T4/L4 1, A100 40 GB 2, 80 GB 4); modeller toplu üretimi paylaşır

import base64, io, os, secrets, subprocess, sys, tarfile, time
t0 = time.time()
PAKET_TARIHI = "2026-11-08 22:16"
PAKET = "..."
import torch
assert torch.cuda.is_available(), "GPU yok: Çalışma zamanı → Çalışma zamanı türünü değiştir → A100 GPU"
print("GPU:", torch.cuda.get_device_name(0), "· paket:", PAKET_TARIHI, flush=True)
if subprocess.run("uv pip install --system -q voxcpm librosa", shell=True).returncode:
    subprocess.run("pip install -q voxcpm librosa", shell=True, check=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAKET))) as t:
    t.extractall("/", filter="data")
sys.path.insert(0, "/content/sesstudio/worker")
import calistir as C
import sesstudio_sunucu as S
n = S.model_sayisi(MODEL_SAYISI)
S.ek_iscileri_baslat(n)   # 2..n. modeller ayrı süreçlerde, bu modelle aynı anda yüklenir
print(C.hazirla("talking-head-sunum"), flush=True)
for _ in range(600):
    if len(S.ek_isciler()) >= n - 1:
        break
    time.sleep(1)
print(f"{1 + len(S.ek_isciler())}/{n} model hazır", flush=True)
SIFRE = SIFRE or secrets.token_urlsafe(8)
url = S.baslat(SIFRE)
from IPython.display import HTML, display
display(HTML(f"""<div style="font:16px system-ui;padding:14px;border:2px solid #E7B65C;border-radius:12px;max-width:640px">
<b>ses-studio hazır</b> ({time.time() - t0:.0f} sn · {1 + len(S.ek_isciler())} model)<br><br>
Sayfa: <a href="{url}" target="_blank" style="font-size:18px">{url}</a><br>
Kullanıcı adı: <code>ses</code> · Şifre: <code>{SIFRE}</code><br><br>
<small>Bu hücre çalıştığı sürece sayfa açık. Bitince sayfadaki "Oturumu kapat"a bas ya da bu hücreyi durdur.</small></div>"""))
S.bekle(bosta_dk=30)
